# Stage 5 — Salary Tier Classification

**Research question:** can salary tier (Low / Mid / High) be reliably predicted from job posting features alone?

This notebook trains four models on an 80/20 stratified split (`random_state=42`):
1. Logistic Regression (baseline)
2. Random Forest (200 trees)
3. XGBoost
4. XGBoost + job-title TF-IDF (top 50 title words), with hyperparameters tuned by cross-validation on the training split

Each is evaluated with macro F1 (target > 0.75), one-vs-rest ROC-AUC (target > 0.80), and a confusion matrix. The best model by macro F1 is saved with its feature importances. The logic lives in `src/classification.py`. To run without the notebook: `python -m src.classification`.

## Step 1 — Setup

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image
from sklearn.model_selection import train_test_split


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import classification as cl

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 2 — Load data

`cleaned_jobs.csv` gives skills, experience level, industry, and the target. The warehouse (`skillmap.db`) adds each job's **state** (from `dim_location`) and its company's **employee_count** (from `dim_company`), joined on `job_id`.

In [ ]:
df = cl.load_model_data(PROJECT_ROOT)
skills = cl.load_skill_vocabulary(PROJECT_ROOT)
y = df["salary_tier"].map(cl.TIER_TO_CODE)
print(f"{len(df):,} jobs; {len(skills)} vocabulary skills")
df["salary_tier"].value_counts()

## Step 3 — Train/test split and feature encoding

The split comes **before** choosing the top states and industries, so those choices use training data only.

| Feature | Encoding |
|---|---|
| `matched_skills` | multi-hot: 100 binary `skill_*` columns |
| `experience_level` | ordinal: Internship 0, Entry 1, Associate (Mid) 2, Mid-Senior (Senior) 3, Director 4, Executive 5, Unknown −1 |
| `company_size` | ordinal from `employee_count`: unknown 0, 1-10 → 1, 11-50 → 2, … 10001+ → 8 |
| state | one-hot: top 20 states + `state_Other` (includes jobs with no state, e.g. "United States") |
| industry | multi-hot: top 15 industries + `industry_Other` (a job can have up to 3 industries) |

The data has no separate "Mid" and "Senior" levels. LinkedIn uses *Associate* and *Mid-Senior level*, which map to 2 and 3.

LinkedIn's own `company_size` column is a 1-7 code rather than a head-count range, so the ranges are built from `employee_count` instead.

In [ ]:
train_df, test_df, y_train, y_test = train_test_split(
    df, y, test_size=cl.TEST_SIZE, stratify=y, random_state=cl.RANDOM_STATE)
top = cl.choose_top_categories(train_df)
X_train = cl.build_features(train_df, skills, top)
X_test = cl.build_features(test_df, skills, top)
print(f"Train {X_train.shape}, test {X_test.shape}")
print("Top states:", top["states"])
print("Top industries:", top["industries"])
X_train.iloc[:5, :8]

## Step 4 — Train and evaluate Models 1-3 (base features)

The results table also reports `train_f1_macro`. Random Forest scores about 0.99 on its own training data, so it memorizes a lot. Among the base-feature models, its test score is still the best.

In [ ]:
models, metrics = cl.train_and_evaluate(X_train, y_train, X_test, y_test)
cl.results_table(metrics)

## Step 5 — Model 4: XGBoost + job-title TF-IDF

Job titles carry pay information that skills don't (*senior*, *engineer*, *technician*, *shift*). Model 4 appends TF-IDF weights for the **50 most frequent title words** (English stop words removed, vectorizer fit on training titles only) to the 139 base features.

XGBoost hyperparameters come from a 3-fold cross-validated grid search (macro F1) on the **training split only**, so the test set plays no part in tuning. The search fits 48 models and takes about 2-3 minutes.

In [ ]:
A_train, A_test, title_names, vectorizer = cl.build_title_features(
    train_df["job_title"], test_df["job_title"], X_train, X_test)
print("Title words:", list(vectorizer.get_feature_names_out()))
models[cl.TITLE_MODEL_NAME], metrics[cl.TITLE_MODEL_NAME], tuning = cl.train_title_model(
    A_train, y_train, A_test, y_test)
print(f"CV macro F1 {tuning['cv_f1_macro']:.3f} with {tuning['best_params']}")

## Step 6 — Compare all four models

The best model by macro F1 is the one saved. Model 4 still misses the F1 > 0.75 target, but it beats the other three on every metric, so it becomes the best model.

In [ ]:
table = cl.results_table(metrics)
table.to_csv(OUTPUT_DIR / "05_classification_results.csv", index=False)
best = table.iloc[0]["model"]
uses_title = best == cl.TITLE_MODEL_NAME
feature_names = title_names if uses_title else list(X_train.columns)
print(f"Best model by macro F1: {best}")
table

In [ ]:
print(metrics[best]["report"])

## Step 7 — Confusion matrices

Rows are the actual tier, and cells show the row percentage and the count. **Mid** is the hardest tier: it's confused with both neighbours, since the tiers are cut from a continuous salary. Low and High are rarely confused with each other.

In [ ]:
cm_path = OUTPUT_DIR / "05_confusion_matrix.png"
cl.plot_confusion_matrices(metrics, cm_path)
Image(filename=str(cm_path), width=1000)

## Step 8 — Feature importance (best model)

These are the top 20 features of the best model. For XGBoost, importance is each feature's share of the total gain; for Random Forest, it's the mean decrease in impurity. Read either as a ranking rather than as effect sizes.

In [ ]:
importances = cl.feature_importance(models[best], feature_names)
fi_path = OUTPUT_DIR / "05_feature_importance.png"
cl.plot_feature_importance(importances, best, fi_path)
Image(filename=str(fi_path), width=750)

## Step 9 — Save the best model and summary

The pickle holds the model together with everything needed to rebuild its features: column order, top states and industries, skill list, encodings, and the fitted title vectorizer if the model uses one.

In [ ]:
cl.save_best_model(models[best], best, feature_names, top, skills, OUTPUT_DIR / "05_best_model.pkl",
                   title_vectorizer=vectorizer if uses_title else None)
summary = cl.build_summary(df, X_train, len(X_train), len(X_test), top, table, metrics, best, importances,
                           title_words=list(vectorizer.get_feature_names_out()), tuning=tuning)
(OUTPUT_DIR / "05_summary.txt").write_text(summary, encoding="utf-8")
print(f"Saved model and {OUTPUT_DIR / '05_summary.txt'}")

## Findings

| Model | Macro F1 | ROC-AUC |
|---|---|---|
| **XGBoost + title TF-IDF (best)** | **0.719** | **0.884** |
| Random Forest | 0.690 | 0.862 |
| XGBoost | 0.667 | 0.847 |
| Logistic Regression | 0.612 | 0.801 |

- **Targets:** the ROC-AUC target (> 0.80) is met by all four models. The **macro F1 target (> 0.75) is not met**; the best is 0.719.
- **What the title words add:** macro F1 rises 0.03 over Random Forest, mostly by getting more Mid jobs right (59% vs 55%).
- **Larger title vocabularies:** tried in exploration, they gain a little more. 2,000 one- and two-word phrases reach about 0.74, still below 0.75.
- **Low vs High is easy.** Only 3-4% of jobs are misclassified across the Low/High gap. Nearly all errors sit at the Mid boundary, which is expected when a continuous salary is cut at the 33rd and 66th percentiles.
- **Strongest signals:**
  - Credential and skill flags: *high school diploma*, *python*, *engineering*, *data entry*, *sql*.
  - Industry: *Retail*, *Hospitality*, *Software Development*.
  - Experience level.
  - Title words: *engineer*, *accountant*, *developer*, *senior*, *shift*.
- **Duplicates were removed first.** Stage 2 drops ads reposted under several `job_id`s. Before that, 15% of test rows had an identical twin in training, which inflated scores.
- **Possible next steps toward 0.75:**
  - A larger title vocabulary.
  - Features from the description text beyond the 100 skills. Any salary figures in the text would need to be removed first to avoid leaking the answer.
  - Treating tier prediction as regression on salary, then binning the prediction.